# Processed data by assay method

The portal API's `AnalysisSet` collection represents processed data. This notebook counts released analysis sets that contain files, grouped by their preferred assay method. Method descriptions come from the portal's `AssayTerm` definitions; if a preferred title maps to multiple terms, their definitions are joined.

In [41]:
from datetime import date
import pandas as pd
import requests
from IPython.display import display
pd.set_option('display.max_colwidth', None)

API_BASE = "https://api.data.igvf.org"
session = requests.Session()

In [42]:
response = session.get(
    API_BASE + "/analysis-sets/@@listing",
    params={"frame": "object", "limit": "all", "status": "released"},
    timeout=(10, 120),
)
response.raise_for_status()
analysis_set_rows = response.json()["@graph"]

assay_response = session.get(
    API_BASE + "/assay-terms/@@listing",
    params={"frame": "object", "limit": "all", "status": "released"},
    timeout=(10, 120),
)
assay_response.raise_for_status()
assay_term_rows = assay_response.json()["@graph"]

method_descriptions = {}
for row in assay_term_rows:
    description = (
        row.get("definition")
        or row.get("description")
        or row.get("summary")
        or row.get("term_name")
    )
    for method in row.get("preferred_assay_titles") or []:
        if description:
            method_descriptions.setdefault(method, set()).add(description)

processed_analysis_sets = [
    row
    for row in analysis_set_rows
    if row.get("status") == "released" and row.get("files")
]

method_rows = [
    {
        "accession": row.get("accession"),
        "method": method,
        "file_set_type": row.get("file_set_type"),
    }
    for row in processed_analysis_sets
    for method in row.get("preferred_assay_titles", [])
]

method_rows = (
    pd.DataFrame(method_rows, columns=["accession", "method", "file_set_type"])
    .drop_duplicates(["accession", "method"])
)
method_rows.head()

,accession,method,file_set_type
0,IGVFDS9402GPQA,10x multiome,intermediate analysis
1,IGVFDS4420UGZN,10x multiome,intermediate analysis
2,IGVFDS2425JMIB,10x snATAC-seq with Scale pre-indexing,intermediate analysis
3,IGVFDS8561TYVW,10x multiome,intermediate analysis
4,IGVFDS3667ORPR,Parse SPLiT-seq,intermediate analysis


In [43]:
method_counts = (
    method_rows.groupby("method", as_index=False)
    .agg(
        analysis_set_count=("accession", "nunique"),
        principal_analysis_sets=(
            "file_set_type",
            lambda values: (values == "principal analysis").sum(),
        ),
        intermediate_analysis_sets=(
            "file_set_type",
            lambda values: (values == "intermediate analysis").sum(),
        ),
    )
    .sort_values(["analysis_set_count", "method"], ascending=[False, True])
    .reset_index(drop=True)
)

method_counts["method_description"] = method_counts["method"].map(
    lambda method: " | ".join(sorted(method_descriptions.get(method, {"No description available."})))
)
method_counts = method_counts[
    [
        "method",
        "analysis_set_count",
        "principal_analysis_sets",
        "intermediate_analysis_sets",
        "method_description"
    ]
]
today_str = date.today().strftime('%Y-%m-%d')
print(f"Data retrieved on {today_str}")
method_counts.to_csv(f"summary_tables/igvf_method_counts_{today_str}.tsv", sep="\t", index=False)
display(method_counts)

Data retrieved on 2026-08-12


,method,analysis_set_count,principal_analysis_sets,intermediate_analysis_sets,method_description
0,10x multiome,1988,506,1482,"A RNA-seq assay that is used for profiling gene expression in individual nuclei with optimized next generation sequencing (NGS) technologies. This allows the RNA-seq profiling of cell types that are more vulnerable to the tissue dissociation process, and that are therefore underrepresented in the final data set in single cell sequencing. | An ATAC-seq assay in which single nuclei are isolated from frozen tissue samples, through a protocol designed to maximally preserve nucleus integrity during sample processing and optimize transposase-mediated fragmentation of chromatin in individual nuclei. | An RNA sequencing assay that uses RNA extracts as input that can be traced to a single cell of origin."
1,Parse SPLiT-seq,535,52,483,"A RNA-seq assay that is used for profiling gene expression in individual nuclei with optimized next generation sequencing (NGS) technologies. This allows the RNA-seq profiling of cell types that are more vulnerable to the tissue dissociation process, and that are therefore underrepresented in the final data set in single cell sequencing. | An RNA sequencing assay that uses RNA extracts as input that can be traced to a single cell of origin."
2,10x snATAC-seq with Scale pre-indexing,144,0,144,"An ATAC-seq assay in which single nuclei are isolated from frozen tissue samples, through a protocol designed to maximally preserve nucleus integrity during sample processing and optimize transposase-mediated fragmentation of chromatin in individual nuclei."
3,10x multiome with MULTI-seq,122,0,122,"A RNA-seq assay that is used for profiling gene expression in individual nuclei with optimized next generation sequencing (NGS) technologies. This allows the RNA-seq profiling of cell types that are more vulnerable to the tissue dissociation process, and that are therefore underrepresented in the final data set in single cell sequencing. | An ATAC-seq assay in which single nuclei are isolated from frozen tissue samples, through a protocol designed to maximally preserve nucleus integrity during sample processing and optimize transposase-mediated fragmentation of chromatin in individual nuclei. | An RNA sequencing assay that uses RNA extracts as input that can be traced to a single cell of origin."
4,SHARE-seq,106,1,105,"A RNA-seq assay that is used for profiling gene expression in individual nuclei with optimized next generation sequencing (NGS) technologies. This allows the RNA-seq profiling of cell types that are more vulnerable to the tissue dissociation process, and that are therefore underrepresented in the final data set in single cell sequencing. | An ATAC-seq assay in which single nuclei are isolated from frozen tissue samples, through a protocol designed to maximally preserve nucleus integrity during sample processing and optimize transposase-mediated fragmentation of chromatin in individual nuclei. | An RNA sequencing assay that uses RNA extracts as input that can be traced to a single cell of origin."
5,scNT-seq2,40,40,0,single cell new transcription sequencing
6,mtscMultiome,38,38,0,An RNA sequencing assay that uses RNA extracts as input that can be traced to a single cell of origin.
7,MPRA,29,15,14,An assay in which multiplexing the construction and interrogation of larger libraries of reporter constructs allows measurement of the transcriptional regulatory activities of thousands to hundreds of thousands of DNA sequences.
8,HiCAR,26,6,20,A chromosome conformation capture assay that detects genome-wide chromosomal interactions. High-throughput techniques are used to sequence the ligated fragments after cross-linking and cutting with restriction enzymes.
9,Pooled Y2H,23,1,22,"An assay that determines interactions between proteins, such as protein-protein binding."


In [44]:
method_counts['analysis_set_count'].sum()

3312

## Processed data by biosample term

In [45]:
def object_id(value):
    if isinstance(value, dict):
        return value.get("@id") or value.get("id")
    return value


def id_variants(value):
    value = object_id(value)
    if not isinstance(value, str):
        return []
    value = value.rstrip("/")
    return [value, value.rsplit("/", 1)[-1]]


def index_objects(rows, fields):
    lookup = {}
    for row in rows:
        for field in fields:
            for key in id_variants(row.get(field)):
                lookup[key] = row
    return lookup


def resolve_object(value, lookup):
    linked_object = {}
    for key in id_variants(value):
        if key in lookup:
            linked_object = lookup[key]
            break
    if isinstance(value, dict):
        return {**linked_object, **value}
    return linked_object


sample_term_response = session.get(
    API_BASE + "/sample-terms/@@listing",
    params={"frame": "object", "limit": "all", "status": "released"},
    timeout=(10, 120),
)
sample_term_response.raise_for_status()
sample_term_rows = sample_term_response.json()["@graph"]
sample_term_lookup = index_objects(sample_term_rows, ["@id", "term_id", "term_name", "name"])

sample_collection_paths = [
    "/in-vitro-systems/@@listing",
    "/multiplexed-samples/@@listing",
    "/primary-cells/@@listing",
    "/technical-samples/@@listing",
    "/tissues/@@listing",
]
sample_rows = []
for path in sample_collection_paths:
    sample_response = session.get(
        API_BASE + path,
        params={"frame": "object", "limit": "all", "status": "released"},
        timeout=(10, 120),
    )
    sample_response.raise_for_status()
    sample_rows.extend(sample_response.json()["@graph"])
sample_lookup = index_objects(sample_rows, ["@id", "accession", "uuid"])

biosample_rows = []
for analysis_set in processed_analysis_sets:
    for sample_ref in analysis_set.get("samples") or []:
        sample = resolve_object(sample_ref, sample_lookup)
        for sample_term_ref in sample.get("sample_terms") or []:
            sample_term = resolve_object(sample_term_ref, sample_term_lookup)
            term_id = sample_term.get("term_id")
            term_name = sample_term.get("term_name") or sample_term.get("name")
            if term_id and term_name:
                biosample_rows.append(
                    {
                        "accession": analysis_set.get("accession"),
                        "biosample_term_id": term_id,
                        "biosample_name": term_name,
                        "file_set_type": analysis_set.get("file_set_type"),
                    }
                )

biosample_rows = pd.DataFrame(
    biosample_rows,
    columns=["accession", "biosample_term_id", "biosample_name", "file_set_type"],
).drop_duplicates(["accession", "biosample_term_id"])

biosample_counts = (
    biosample_rows.groupby("biosample_term_id", as_index=False)
    .agg(
        biosample_name=("biosample_name", "first"),
        analysis_set_count=("accession", "nunique"),
        principal_analysis_sets=(
            "file_set_type",
            lambda values: (values == "principal analysis").sum(),
        ),
        intermediate_analysis_sets=(
            "file_set_type",
            lambda values: (values == "intermediate analysis").sum(),
        ),
    )
    .sort_values(["analysis_set_count", "biosample_name"], ascending=[False, True])
    .reset_index(drop=True)
)
biosample_counts = biosample_counts[
    [
        "biosample_term_id",
        "biosample_name",
        "analysis_set_count",
        "principal_analysis_sets",
        "intermediate_analysis_sets",
    ]
]
print(f"Data retrieved on {today_str}")
biosample_counts.to_csv(f"summary_tables/igvf_biosample_counts_{today_str}.tsv", sep="\t", index=False)
display(biosample_counts)

Data retrieved on 2026-08-12


,biosample_term_id,biosample_name,analysis_set_count,principal_analysis_sets,intermediate_analysis_sets
0,UBERON:0002740,posterior cingulate gyrus,1349,109,1240
1,UBERON:0001874,putamen,1331,111,1220
2,UBERON:0002771,middle temporal gyrus,1295,107,1188
3,UBERON:0002038,substantia nigra,1289,109,1180
4,UBERON:0002129,cerebellar cortex,1099,109,990
...,...,...,...,...,...
72,UBERON:0000956,cerebral cortex,1,1,0
73,CL:0011108,colon epithelial cell,1,0,1
74,UBERON:0000397,colonic epithelium,1,0,1
75,CL:0000235,macrophage,1,1,0


In [46]:
biosample_counts.head(50)

,biosample_term_id,biosample_name,analysis_set_count,principal_analysis_sets,intermediate_analysis_sets
0,UBERON:0002740,posterior cingulate gyrus,1349,109,1240
1,UBERON:0001874,putamen,1331,111,1220
2,UBERON:0002771,middle temporal gyrus,1295,107,1188
3,UBERON:0002038,substantia nigra,1289,109,1180
4,UBERON:0002129,cerebellar cortex,1099,109,990
5,UBERON:0002371,bone marrow,228,1,227
6,NTR:0000646,left cerebral cortex,197,30,167
7,CL:2000001,peripheral blood mononuclear cell,145,39,106
8,UBERON:0002369,adrenal gland,122,10,112
9,UBERON:0001388,gastrocnemius,118,8,110


## Processed data containing variant effects

The following tables use only released analysis sets that contain at least one tabular file whose `content_type` contains `variant effects`, including values such as `coding variant effects`.

In [47]:
tabular_file_response = session.get(
    API_BASE + "/tabular-files/@@listing",
    params={"frame": "object", "limit": "all"},
    timeout=(10, 120),
)
tabular_file_response.raise_for_status()
tabular_file_rows = tabular_file_response.json()["@graph"]
tabular_file_lookup = index_objects(tabular_file_rows, ["@id", "accession"])

def contains_variant_effects_file(analysis_set):
    for file_ref in analysis_set.get("files") or []:
        file = resolve_object(file_ref, tabular_file_lookup)
        content_type = file.get("content_type")
        if isinstance(content_type, str) and "variant effects" in content_type.lower():
            return True
    return False

variant_effect_analysis_sets = [
    analysis_set
    for analysis_set in processed_analysis_sets
    if contains_variant_effects_file(analysis_set)
]
variant_effect_accessions = {
    analysis_set.get("accession") for analysis_set in variant_effect_analysis_sets
}
variant_effect_types = sorted({
    file.get("content_type")
    for file in tabular_file_rows
    if isinstance(file.get("content_type"), str)
    and "variant effects" in file["content_type"].lower()
})
print("Matched file content types:", variant_effect_types)
print("Analysis sets retained:", len(variant_effect_analysis_sets))

Matched file content types: ['aggregated calibrated coding variant effects', 'calibrated coding variant effects', 'coding variant effects', 'reporter genomic variant effects', 'variant effects']
Analysis sets retained: 81


In [48]:
variant_effect_method_rows = method_rows[
    method_rows["accession"].isin(variant_effect_accessions)
].copy()

variant_effect_method_counts = (
    variant_effect_method_rows.groupby("method", as_index=False)
    .agg(
        analysis_set_count=("accession", "nunique"),
        principal_analysis_sets=(
            "file_set_type",
            lambda values: (values == "principal analysis").sum(),
        ),
        intermediate_analysis_sets=(
            "file_set_type",
            lambda values: (values == "intermediate analysis").sum(),
        ),
    )
    .sort_values(["analysis_set_count", "method"], ascending=[False, True])
    .reset_index(drop=True)
)
variant_effect_method_counts["method_description"] = variant_effect_method_counts["method"].map(
    lambda method: " | ".join(sorted(method_descriptions.get(method, {"No description available."})))
)
variant_effect_method_counts = variant_effect_method_counts[
    [
        "method",
        "analysis_set_count",
        "principal_analysis_sets",
        "intermediate_analysis_sets",
        "method_description",
    ]
]
display(variant_effect_method_counts)

,method,analysis_set_count,principal_analysis_sets,intermediate_analysis_sets,method_description
0,STARR-seq,15,15,0,"An RNA-seq assay that identifies the sequences that act as transcriptional enhancers in a direct, quantitative, and genome-wide manner from sheared genomic DNA."
1,Variant-EFFECTS,13,13,0,An in vitro CRISPR screen assay that uses flow cytometry to detect induced changes in gene expression of the targeted genes.
2,lentiMPRA,9,9,0,An assay in which multiplexing the construction and interrogation of larger libraries of reporter constructs allows measurement of the transcriptional regulatory activities of thousands to hundreds of thousands of DNA sequences.
3,MAVE,8,8,0,yeast-based functional complementation assay
4,CRISPR FACS screen,7,7,0,An in vitro CRISPR screen assay that uses flow cytometry to detect induced changes in gene expression of the targeted genes.
5,MPRA,7,7,0,An assay in which multiplexing the construction and interrogation of larger libraries of reporter constructs allows measurement of the transcriptional regulatory activities of thousands to hundreds of thousands of DNA sequences.
6,SGE,7,7,0,Saturation genome editing
7,VAMP-seq (MultiSTEP),5,5,0,Variant abundance by massively parallel sequencing MultiSTEP
8,SGE-RNA,4,4,0,Saturation genome editing
9,VAMP-seq,3,3,0,Variant abundance by massively parallel sequencing


In [49]:
variant_effect_biosample_rows = biosample_rows[
    biosample_rows["accession"].isin(variant_effect_accessions)
].copy().merge(
    variant_effect_method_rows[["accession", "method"]],
    on="accession",
    how="left",
)

variant_effect_biosample_counts = (
    variant_effect_biosample_rows.groupby("biosample_term_id", as_index=False)
    .agg(
        biosample_name=("biosample_name", "first"),
        analysis_set_count=("accession", "nunique"),
        principal_analysis_sets=(
            "file_set_type",
            lambda values: (values == "principal analysis").sum(),
        ),
        intermediate_analysis_sets=(
            "file_set_type",
            lambda values: (values == "intermediate analysis").sum(),
        ),
        methods=("method", lambda values: sorted(set(values.dropna()))),
    )
    .sort_values(["analysis_set_count", "biosample_name"], ascending=[False, True])
    .reset_index(drop=True)
)
variant_effect_biosample_counts = variant_effect_biosample_counts[
    [
        "biosample_term_id",
        "biosample_name",
        "analysis_set_count",
        "principal_analysis_sets",
        "intermediate_analysis_sets",
        "methods",
    ]
]
display(variant_effect_biosample_counts)

,biosample_term_id,biosample_name,analysis_set_count,principal_analysis_sets,intermediate_analysis_sets,methods
0,EFO:0002067,K562,15,15,0,[STARR-seq]
1,EFO:0001253,THP-1,10,10,0,"[Variant-EFFECTS, lentiMPRA]"
2,EFO:0001187,HepG2,9,9,0,"[CRISPR FACS screen, MPRA, electroporated MPRA, lentiMPRA]"
3,EFO:0007598,HAP-1,7,11,0,"[SGE, SGE-RNA]"
4,CL:0000000,cell,6,6,0,[MAVE]
5,EFO:0022564,HEK-293F,5,5,0,[VAMP-seq (MultiSTEP)]
6,EFO:0009747,GM25256 (WTC-11),4,4,0,[lentiMPRA]
7,EFO:0001082,HEK293T,4,4,0,"[VAMP-seq, lentiMPRA]"
8,EFO:0002796,Jurkat,4,4,0,[Variant-EFFECTS]
9,CLO:0007938,NALM-6 cell,3,3,0,[CRISPR FACS screen]
